# 10 — Document Intelligence

## What
Document intelligence models extract structured information from documents that contain both text and layout — invoices, forms, medical records, research papers, legal contracts. The key models are layout-aware transformers (LayoutLM, Donut) that encode text tokens together with their 2D bounding box positions.

## Why
Pure OCR loses layout information that is semantically critical. The value of 'Total: $1,450' depends on its position relative to the table column header. LayoutLM and Donut achieve near-human performance on information extraction by making 2D position a first-class input.

## Community context
LayoutLM (Microsoft, 2020) added 2D position embeddings to BERT. LayoutLMv2 added image features. LayoutLMv3 unified text, layout, and image encoding. Donut (Kim et al., 2021) takes a completely different approach: end-to-end OCR-free document understanding using a ViT encoder + autoregressive decoder, reading the document as a raw image.

In [ ]:
# LayoutLM-style: text token + 2D bounding box position encoding
import numpy as np

class Layout2DPositionEmbedding:
    """
    Encodes each token's bounding box (x0, y0, x1, y1, width, height)
    as learnable embeddings added to the token embedding.
    Coordinates are normalised to [0, 1000] as in LayoutLM.
    """
    def __init__(self, embed_dim=64, max_pos=1001):
        self.embed_dim = embed_dim
        # Separate embedding tables for each coordinate dimension
        self.x_embed = np.random.randn(max_pos, embed_dim//6) * 0.02
        self.y_embed = np.random.randn(max_pos, embed_dim//6) * 0.02
        self.w_embed = np.random.randn(max_pos, embed_dim//6) * 0.02
        self.h_embed = np.random.randn(max_pos, embed_dim//6) * 0.02
        # Remainder goes to x1, y1 embeddings
        rem = embed_dim - 4*(embed_dim//6)
        self.x1_embed = np.random.randn(max_pos, rem//2) * 0.02
        self.y1_embed = np.random.randn(max_pos, rem - rem//2) * 0.02

    def __call__(self, bbox):
        """
        bbox: dict with x0,y0,x1,y1 normalised to [0,1000]
        Returns: layout position embedding (embed_dim,)
        """
        x0,y0,x1,y1 = int(bbox['x0']), int(bbox['y0']), int(bbox['x1']), int(bbox['y1'])
        w = min(x1-x0, 1000)
        h = min(y1-y0, 1000)
        parts = [
            self.x_embed[x0], self.y_embed[y0],
            self.x1_embed[x1], self.y1_embed[y1],
            self.w_embed[w], self.h_embed[h],
        ]
        return np.concatenate(parts)

class LayoutLMTokenEncoder:
    def __init__(self, vocab_size=1000, embed_dim=64):
        self.text_embed = np.random.randn(vocab_size, embed_dim) * 0.02
        self.layout_embed = Layout2DPositionEmbedding(embed_dim)

    def encode_token(self, token_id, bbox):
        text  = self.text_embed[token_id % 1000]
        layout = self.layout_embed(bbox)
        return text + layout

# Simulate an invoice document with OCR output
invoice_tokens = [
    {'text': 'INVOICE',       'token_id': 42,  'bbox': {'x0': 50, 'y0': 20, 'x1': 200, 'y1': 50}},
    {'text': '#12345',        'token_id': 12,  'bbox': {'x0': 210,'y0': 20, 'x1': 350, 'y1': 50}},
    {'text': 'Item',          'token_id': 88,  'bbox': {'x0': 50, 'y0': 100,'x1': 150, 'y1': 125}},
    {'text': 'Quantity',      'token_id': 99,  'bbox': {'x0': 200,'y0': 100,'x1': 350, 'y1': 125}},
    {'text': 'Price',         'token_id': 77,  'bbox': {'x0': 400,'y0': 100,'x1': 500, 'y1': 125}},
    {'text': 'Widget',        'token_id': 55,  'bbox': {'x0': 50, 'y0': 140,'x1': 150, 'y1': 165}},
    {'text': '5',             'token_id': 5,   'bbox': {'x0': 200,'y0': 140,'x1': 250, 'y1': 165}},
    {'text': '$290.00',       'token_id': 290, 'bbox': {'x0': 400,'y0': 140,'x1': 500, 'y1': 165}},
    {'text': 'Total',         'token_id': 111, 'bbox': {'x0': 300,'y0': 200,'x1': 400, 'y1': 225}},
    {'text': '$1450.00',      'token_id': 145, 'bbox': {'x0': 400,'y0': 200,'x1': 550, 'y1': 225}},
]

encoder = LayoutLMTokenEncoder(embed_dim=64)
all_embeds = []
for tok in invoice_tokens:
    emb = encoder.encode_token(tok['token_id'], tok['bbox'])
    all_embeds.append(emb)

print('LayoutLM Token Encoding (invoice):')
for tok, emb in zip(invoice_tokens, all_embeds):
    x0, y1 = tok['bbox']['x0'], tok['bbox']['y1']
    print(f'  "{tok["text"]:<12}" bbox=({x0:3d},{y1:3d}) embed[0:4]={emb[:4].round(3)}')

token_matrix = np.array(all_embeds)
print(f'\nToken embedding matrix: {token_matrix.shape}')
print('2D position info allows model to understand table structure and field relationships')

## Key-Value Extraction

Document intelligence frequently requires extracting specific fields (key-value pairs) from documents. A token classification head trained on labelled invoice/form data predicts BIO tags (Beginning/Inside/Outside) for each field.

In [ ]:
# Simulated key-value extraction with BIO tagging
import numpy as np

FIELD_TYPES = ['O', 'B-TOTAL', 'I-TOTAL', 'B-INVOICE_NUM', 'I-INVOICE_NUM',
               'B-DATE', 'I-DATE', 'B-ITEM', 'I-ITEM', 'B-PRICE', 'I-PRICE']

# Ground truth labels for our invoice
gt_labels = [
    'O',             # INVOICE
    'B-INVOICE_NUM', # #12345
    'O',             # Item
    'O',             # Quantity
    'O',             # Price
    'B-ITEM',        # Widget
    'O',             # 5
    'B-PRICE',       # $290.00
    'O',             # Total
    'B-TOTAL',       # $1450.00
]

# Simulate classifier scores (token_matrix -> FIELD_TYPES)
np.random.seed(3)
W_cls = np.random.randn(64, len(FIELD_TYPES)) * 0.1
logits = token_matrix @ W_cls  # (N_tokens, N_fields)

# Add signal: make ground truth label score slightly higher
for i, gt in enumerate(gt_labels):
    gt_idx = FIELD_TYPES.index(gt)
    logits[i, gt_idx] += 2.0

def softmax(x, axis=-1):
    e = np.exp(x - x.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)

probs = softmax(logits)
pred_labels = [FIELD_TYPES[np.argmax(probs[i])] for i in range(len(invoice_tokens))]

print('Key-Value Extraction Results:')
print(f'{"Token":<14} {"Predicted":<18} {"GT":<18} {"Match"}')
correct = 0
for tok, pred, gt in zip(invoice_tokens, pred_labels, gt_labels):
    match = pred == gt
    correct += match
    print(f'  {tok["text"]:<12}  {pred:<18} {gt:<18} {"OK" if match else "WRONG"}')
print(f'\nToken accuracy: {correct}/{len(gt_labels)} = {correct/len(gt_labels):.1%}')

# Extract fields
fields = {}
for tok, pred in zip(invoice_tokens, pred_labels):
    if pred.startswith('B-'):
        field = pred[2:]
        fields[field] = tok['text']
print(f'\nExtracted fields: {fields}')